In [7]:
import numpy as np
import datetime
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt
from fredapi import Fred

In [8]:
np.random.seed(42)

# define the tickers that make up the portfolio, the start date and end date.
tickers = ["EXV1.DE", "EXV2.DE", "EXV3.DE", "EXV5.DE", "EXV6.DE"]
start_date = "2014-06-06"
end_date = "2024-05-27"

# download the data for each ticker (Close and Volume)
stocks_data = yf.download(tickers, start=start_date, end=end_date, interval='1d')
stocks_data["Close"].to_csv("stocks_data_adj_close.csv")

# resample data on a weekly basis (using Friday's closing)
weekly_adj_close = stocks_data['Close'].resample('W-FRI').last()
weekly_volume = stocks_data['Volume'].resample('W-FRI').sum()

# calculate the logarithm of weekly returns and weekly volume change
log_returns = np.log(weekly_adj_close / weekly_adj_close.shift(1))
log_returns.dropna(inplace=True)
log_volume_change = np.log(weekly_volume / weekly_volume.shift(1))
log_volume_change.dropna(inplace=True)

# combine logarithmic returns and logarithmic changes in volume
features = log_returns.join(log_volume_change, lsuffix='_log_returns', rsuffix='_log_volume_change')

# add columns by year, month, and week
features['Year'] = features.index.year
features['Month'] = features.index.month
features['Week'] = features.index.isocalendar().week

features.head()

[*********************100%***********************]  5 of 5 completed


Ticker,EXV1.DE_log_returns,EXV2.DE_log_returns,EXV3.DE_log_returns,EXV5.DE_log_returns,EXV6.DE_log_returns,EXV1.DE_log_volume_change,EXV2.DE_log_volume_change,EXV3.DE_log_volume_change,EXV5.DE_log_volume_change,EXV6.DE_log_volume_change,Year,Month,Week
Date,,,,,,,,,,,,,
2014-06-13,-0.011657,-0.001660,0.001065,-0.018531,-0.010385,1.322026,0.255979,4.518050,1.224688,3.175903,2014,6,24
2014-06-20,-0.021527,-0.008005,0.007070,0.007492,0.020664,-1.940665,-0.685042,-4.943396,-1.810953,-1.004738,2014,6,25
2014-06-27,-0.031544,-0.005036,-0.021721,-0.020887,-0.008598,0.971962,1.039662,3.945714,1.650998,-0.381296,2014,6,26
2014-07-04,0.012288,0.000673,0.024183,0.021269,0.052791,0.331784,-0.544316,-0.167457,-1.039380,0.828782,2014,7,27
2014-07-11,-0.040179,-0.030740,-0.036133,-0.039213,-0.023715,1.274162,0.929091,0.554028,0.796088,1.124818,2014,7,28


In [9]:
np.random.seed(42)

# download the data for Gold and Oil
tickers = ["GC=F", "CL=F"]
gold_oil_data = yf.download(tickers, start=start_date, end=end_date, interval='1d')

# resample data on a weekly basis (using Friday's closing)
weekly_adj_close = gold_oil_data['Close'].resample('W-FRI').last()
weekly_volume = gold_oil_data['Volume'].resample('W-FRI').sum()

# calculate the logarithm of weekly returns and weekly volume change
log_returns = np.log(weekly_adj_close / weekly_adj_close.shift(1))
log_returns.dropna(inplace=True)
log_volume_change = np.log(weekly_volume / weekly_volume.shift(1))
log_volume_change.dropna(inplace=True)

# combine logarithmic returns and logarithmic changes in volume
gold_oil_features = log_returns.join(log_volume_change, lsuffix='_log_returns', rsuffix='_log_volume_change')

# add columns by year, month, and week
gold_oil_features['Year'] = features.index.year
gold_oil_features['Month'] = features.index.month
gold_oil_features['Week'] = features.index.isocalendar().week

# combine new macroeconomic data (gold price and oil price) with previously calculated features.
features = pd.merge(features, gold_oil_features, on=['Year', 'Month', 'Week'], how='left')
features.head()

[*********************100%***********************]  2 of 2 completed


Ticker,EXV1.DE_log_returns,EXV2.DE_log_returns,EXV3.DE_log_returns,EXV5.DE_log_returns,EXV6.DE_log_returns,EXV1.DE_log_volume_change,EXV2.DE_log_volume_change,EXV3.DE_log_volume_change,EXV5.DE_log_volume_change,EXV6.DE_log_volume_change,Year,Month,Week,CL=F_log_returns,GC=F_log_returns,CL=F_log_volume_change,GC=F_log_volume_change
0,-0.011657,-0.001660,0.001065,-0.018531,-0.010385,1.322026,0.255979,4.518050,1.224688,3.175903,2014,6,24,0.040565,0.017104,1.857315,1.374413
1,-0.021527,-0.008005,0.007070,0.007492,0.020664,-1.940665,-0.685042,-4.943396,-1.810953,-1.004738,2014,6,25,0.003268,0.032823,-0.266786,0.356476
2,-0.031544,-0.005036,-0.021721,-0.020887,-0.008598,0.971962,1.039662,3.945714,1.650998,-0.381296,2014,6,26,-0.014273,0.002125,-0.048384,0.145546
3,0.012288,0.000673,0.024183,0.021269,0.052791,0.331784,-0.544316,-0.167457,-1.039380,0.828782,2014,7,27,-0.016016,0.001061,-0.087460,-0.852267
4,-0.040179,-0.030740,-0.036133,-0.039213,-0.023715,1.274162,0.929091,0.554028,0.796088,1.124818,2014,7,28,-0.031532,0.012494,0.348549,-1.425704


In [10]:
# upload data on Eurozone interest rates (source: https://fred.stlouisfed.org/series/IR3TIB01EZM156N) 
eu_interest_rates = pd.read_csv("eu_interest_rates.csv")
eu_interest_rates['Date'] = pd.to_datetime(eu_interest_rates['Date'])

# upload data on the HCPI harmonized consumer price index (source: https://ec.europa.eu/eurostat/databrowser/view/prc_hicp_midx__custom_11923354/default/table?lang=en) 
eu_hcpi = pd.read_csv("eu_hcpi.csv")
eu_hcpi['Date'] = pd.to_datetime(eu_hcpi['Date'] + '-01')

# merge the interest rates and the value of the HCPI index into a single DataFrame.
interest_rates_and_hcpi = pd.merge(eu_interest_rates, eu_hcpi, on='Date', how='left')
interest_rates_and_hcpi["Date"] = pd.to_datetime(interest_rates_and_hcpi["Date"])
interest_rates_and_hcpi = interest_rates_and_hcpi[interest_rates_and_hcpi["Date"] >= pd.to_datetime('2014-05-01')]

# add the year and month columns
interest_rates_and_hcpi['Year'] = interest_rates_and_hcpi['Date'].dt.year
interest_rates_and_hcpi['Month'] = interest_rates_and_hcpi['Date'].dt.month

# calculate the logarithm of the HCPI changes and standardize the interest rates.
interest_rates_and_hcpi['log_change_eu_hcpi'] = np.log(interest_rates_and_hcpi['eu_hcpi'] / interest_rates_and_hcpi['eu_hcpi'].shift(1))
interest_rates_and_hcpi["eu_interest_rates_z_score"]= (eu_interest_rates["eu_interest_rates"] - eu_interest_rates["eu_interest_rates"].mean()) / eu_interest_rates["eu_interest_rates"].std()

# remove the original interest_rates and hcpi columns and rows with NaN values.
interest_rates_and_hcpi = interest_rates_and_hcpi.drop(columns=['eu_hcpi', 'eu_interest_rates'])
interest_rates_and_hcpi = interest_rates_and_hcpi.dropna()

# combine new macroeconomic data (euro area interest rates and euro area inflation rate) with previously calculated features.
features = pd.merge(features, interest_rates_and_hcpi, on=['Year', 'Month'], how='left')
features = features.drop(columns=['Date', 'Year', 'Week'])
features.head()

,EXV1.DE_log_returns,EXV2.DE_log_returns,EXV3.DE_log_returns,EXV5.DE_log_returns,EXV6.DE_log_returns,EXV1.DE_log_volume_change,EXV2.DE_log_volume_change,EXV3.DE_log_volume_change,EXV5.DE_log_volume_change,EXV6.DE_log_volume_change,Month,CL=F_log_returns,GC=F_log_returns,CL=F_log_volume_change,GC=F_log_volume_change,log_change_eu_hcpi,eu_interest_rates_z_score
0,-0.011657,-0.001660,0.001065,-0.018531,-0.010385,1.322026,0.255979,4.518050,1.224688,3.175903,6,0.040565,0.017104,1.857315,1.374413,0.000998,-0.057502
1,-0.021527,-0.008005,0.007070,0.007492,0.020664,-1.940665,-0.685042,-4.943396,-1.810953,-1.004738,6,0.003268,0.032823,-0.266786,0.356476,0.000998,-0.057502
2,-0.031544,-0.005036,-0.021721,-0.020887,-0.008598,0.971962,1.039662,3.945714,1.650998,-0.381296,6,-0.014273,0.002125,-0.048384,0.145546,0.000998,-0.057502
3,0.012288,0.000673,0.024183,0.021269,0.052791,0.331784,-0.544316,-0.167457,-1.039380,0.828782,7,-0.016016,0.001061,-0.087460,-0.852267,-0.005200,-0.083717
4,-0.040179,-0.030740,-0.036133,-0.039213,-0.023715,1.274162,0.929091,0.554028,0.796088,1.124818,7,-0.031532,0.012494,0.348549,-1.425704,-0.005200,-0.083717


In [11]:
# using the Month column I get 12 dummy variables
features = pd.get_dummies(features, columns=['Month'], prefix='Month')
# convert dummy columns from True/False to 0/1
dummy_columns = features.filter(like='Month_').columns
features[dummy_columns] = features[dummy_columns].astype(int)

features = features.dropna()
features.head()

,EXV1.DE_log_returns,EXV2.DE_log_returns,EXV3.DE_log_returns,EXV5.DE_log_returns,EXV6.DE_log_returns,EXV1.DE_log_volume_change,EXV2.DE_log_volume_change,EXV3.DE_log_volume_change,EXV5.DE_log_volume_change,EXV6.DE_log_volume_change,...,Month_3,Month_4,Month_5,Month_6,Month_7,Month_8,Month_9,Month_10,Month_11,Month_12
0,-0.011657,-0.001660,0.001065,-0.018531,-0.010385,1.322026,0.255979,4.518050,1.224688,3.175903,...,0,0,0,1,0,0,0,0,0,0
1,-0.021527,-0.008005,0.007070,0.007492,0.020664,-1.940665,-0.685042,-4.943396,-1.810953,-1.004738,...,0,0,0,1,0,0,0,0,0,0
2,-0.031544,-0.005036,-0.021721,-0.020887,-0.008598,0.971962,1.039662,3.945714,1.650998,-0.381296,...,0,0,0,1,0,0,0,0,0,0
3,0.012288,0.000673,0.024183,0.021269,0.052791,0.331784,-0.544316,-0.167457,-1.039380,0.828782,...,0,0,0,0,1,0,0,0,0,0
4,-0.040179,-0.030740,-0.036133,-0.039213,-0.023715,1.274162,0.929091,0.554028,0.796088,1.124818,...,0,0,0,0,1,0,0,0,0,0


In [12]:
# check for the presence of any NaN values
nan_counts = features.isna().sum()
print(nan_counts)

# save features to a csv
features.to_csv("features.csv", index= False)

EXV1.DE_log_returns          0
EXV2.DE_log_returns          0
EXV3.DE_log_returns          0
EXV5.DE_log_returns          0
EXV6.DE_log_returns          0
EXV1.DE_log_volume_change    0
EXV2.DE_log_volume_change    0
EXV3.DE_log_volume_change    0
EXV5.DE_log_volume_change    0
EXV6.DE_log_volume_change    0
CL=F_log_returns             0
GC=F_log_returns             0
CL=F_log_volume_change       0
GC=F_log_volume_change       0
log_change_eu_hcpi           0
eu_interest_rates_z_score    0
Month_1                      0
Month_2                      0
Month_3                      0
Month_4                      0
Month_5                      0
Month_6                      0
Month_7                      0
Month_8                      0
Month_9                      0
Month_10                     0
Month_11                     0
Month_12                     0
dtype: int64
